# Уголок Антипина: сверка с Дойчем–Канделасом и строгий расчёт уголка конечных размеров

**Часть I** — сверка трёх формул для силы на грани уголка: Антипин (arXiv:1702.0328, ф. (3)), последовательное PFA, точное решение Дойча–Канделаса (Phys. Rev. D 20, 3063 (1979), ф. (5.42), (5.44)).
**Часть II** — теория строгого расчёта уголка конечных размеров методом функции Грина (от постановки до формулы силы на каждый элемент поверхности).
**Часть III** — численное решение: сила на каждый элемент; раскладка «грани / концы у вершины / внешние концы»; сходимость.

Единицы $\hbar=c=1$. Ядро SageMath 9.6, код — Python + numpy (препарсер Sage отключается в первой ячейке).
Источники лежат в `articles/casimir/ref/` и `articles/references/Casimir/Antipin/`.

In [ ]:
try:
    preparser(False)
except NameError:
    pass
import numpy as np, time
pi = np.pi
def check(name, residual, tol):
    r = float(abs(residual))
    print(('OK  ' if r < tol else 'FAIL') + '  %-66s residual = %.3e (tol %.0e)' % (name, r, tol))

## Часть I. Сверка формул для давления на грани

Обозначения: $\alpha$ — **полуугол** уголка, $r$ — расстояние от вершины вдоль грани, $b$ — длина уголка вдоль оси.

**Дойч–Канделас (5.44)** — ЭМ поле внутри идеально проводящего клина с полным углом $\theta=2\alpha$:
$$\langle T_\mu{}^\nu\rangle=\frac{f_{\rm EM}(\theta)}{720\pi^2r^4}\,\mathrm{diag}(1,1,-3,1),\qquad f_{\rm EM}(\theta)=\Big(\frac{\pi^2}{\theta^2}+11\Big)\Big(\frac{\pi^2}{\theta^2}-1\Big).$$
Давление на грань (компонента $\theta\theta$): $p=\dfrac{f(\theta)}{240\pi^2r^4}$. Снаружи грани — клин с углом $2\pi-\theta$.
Проекция на биссектрису — множитель $\sin\alpha$, две грани, $\int r^{-4}dr=\tfrac13(r_{\min}^{-3}-r_{\max}^{-3})$:
$$F_{\rm DC}=\frac{b\sin\alpha\,[f(2\alpha)-f(2\pi-2\alpha)]}{360\pi^2}\Big(\frac1{r_{\min}^3}-\frac1{r_{\max}^3}\Big).$$

**Антипин, ф. (3)** — в переменных «расстояние между сторонами» $L=2r\sin\alpha$:
$F=\dfrac{\pi^2 b}{1440}\dfrac{\cos2\alpha}{\cos^4\alpha}\Big(\dfrac1{L_{\min}^3}-\dfrac1{L_{\max}^3}\Big)$.

**Последовательное PFA** (зазор $a(z)=2z\,\mathrm{tg}\,\alpha$, давление $\pi^2/240a^4$ по нормали, $dS=b\,dz/\cos\alpha$):
$$F_{\rm PFA}=\frac{\pi^2 b}{5760\,\sin^3\alpha}\Big(\frac1{r_{\min}^3}-\frac1{r_{\max}^3}\Big).$$

Ниже все три приведены к одному коэффициенту $K$ при $b\,(L_{\min}^{-3}-L_{\max}^{-3})$; ожидание — при $\alpha\to0$ PFA и Дойч–Канделас совпадают (это предел пластин Казимира, о котором пишут сами Дойч и Канделас, стр. 3064).

In [ ]:
fEM = lambda t: (pi**2/t**2 + 11) * (pi**2/t**2 - 1)
def K_DC(a):                  # к b*(1/L^3), L = 2 r sin a
    return 8*np.sin(a)**3 * np.sin(a) * (fEM(2*a) - fEM(2*pi - 2*a)) / (360*pi**2)
def K_DC_in(a):
    return 8*np.sin(a)**4 * fEM(2*a) / (360*pi**2)
def K_A(a):
    return pi**2/1440 * np.cos(2*a) / np.cos(a)**4
def K_PFA(a):
    return pi**2/(5760*np.sin(a)**3) * 8*np.sin(a)**3
print(' alpha    Антипин(3)   PFA        DC изнутри   DC изн.+снар.   DC/Антипин')
for ad in (1, 3, 10, 20, 30, 40, 45):
    a = np.radians(ad)
    print('%4d°   %.5f   %.5f   %.5f     %.5f        %6.3f' % (ad, K_A(a), K_PFA(a), K_DC_in(a), K_DC(a), K_DC(a)/K_A(a) if ad < 45 else np.inf))
a = np.radians(0.5)
check('alpha->0: PFA = DC (предел пластин Казимира)', K_PFA(a)/K_DC(a) - 1, 2e-3)
check('alpha->0: DC/Антипин = 2', K_DC(a)/K_A(a) - 2, 2e-3)
# проверка интегрирования PFA численно
al = np.radians(15); z = np.linspace(0.2*np.cos(al), np.cos(al), 400001)
num = 2*np.trapezoid(pi**2/(240*(2*z*np.tan(al))**4) * np.sin(al)/np.cos(al), z)
check('PFA: численный интеграл = pi^2/(5760 sin^3a)(1/r_min^3-1/r_max^3)', num/(pi**2/(5760*np.sin(al)**3)*(1/0.2**3-1)) - 1, 1e-6)
# ошибки приложения D 2024 г.: (D.5) = pi^2/640 cos a/sin^4 a  против  pi^2/(5760 sin^3 a)
for ad in (15, 45):
    al = np.radians(ad); old = pi**2/640*np.cos(al)/np.sin(al)**4; new = pi**2/(5760*np.sin(al)**3)
    check('прил. D 2024: (D.5)/верное = 9 ctg(alpha), alpha=%d°' % ad, old/new - 9/np.tan(al), 1e-9)

**Итог части I.** Нормальное давление внутрь уголка и закон $1/L^3$ у Антипина подтверждаются точной теорией; при малых углах его коэффициент в 2 раза меньше точного (последовательное PFA совпадает с точным). Множитель $\cos2\alpha/\cos^4\alpha$ — артефакт вывода. В приложении D статьи 2024 г. две ошибки ($dS=b\,dz$ вместо $b\,dz/\cos\alpha$ и множитель $3$ вместо $\tfrac13$), дающие завышение в $9\,\mathrm{ctg}\,\alpha$ раз.

Всё это — давление на **грани бесконечного клина**. Вопрос о полной силе на уголок решает расчёт **конечного** тела: части II–III.

## Часть II. Теория: уголок конечных размеров методом функции Грина

### II.1 Модель
* Уголок бесконечен вдоль оси $z$ (сила на единицу длины $b$), в плоскости $(x,y)$ — два отрезка (крылья) от $r_0$ до $r_1$ под углами $\pm\alpha$ к оси $y$; вершина в начале координат, раствор — в сторону $+y$. При $r_0>0$ крылья не касаются (у вершины зазор $2r_0\sin\alpha$).
* Поле — безмассовый скаляр с условием Дирихле на крыльях (нулевая толщина). Для геометрии, однородной по $z$, ЭМ поле идеального проводника точно распадается на TM‑ (Дирихле) и TE‑ (Нейман) части; здесь считается TM‑часть.

### II.2 Энергия через функцию Грина
Евклидова функция Грина свободного пространства после фурье‑преобразования по $z$ и по мнимой частоте $\xi$:
$$G_0(\mathbf r,\mathbf r';\kappa)=\frac{1}{2\pi}K_0(\kappa|\mathbf r-\mathbf r'|),\qquad \kappa^2=\xi^2+k_z^2 .$$
Условие Дирихле на кривых $\Gamma$ реализуется потенциалом простого слоя: $\varphi(\mathbf r)=\int_\Gamma G_0(\mathbf r,\mathbf s)\sigma(\mathbf s)ds$, а $\varphi|_\Gamma=0$ даёт оператор $\hat S(\kappa)$: $(\hat S\sigma)(\mathbf s)=\int_\Gamma G_0(\mathbf s,\mathbf s')\sigma(\mathbf s')ds'$.
Энергия нулевых колебаний (формула Ллойда / TGTG, Rahi–Emig–Graham–Jaffe–Kardar 2009) на единицу длины:
$$\mathcal E=\frac{1}{2\pi}\int_0^\infty d\xi\int_{-\infty}^{\infty}\frac{dk_z}{2\pi}\,\ln\frac{\det\hat S}{\det\hat S_1\det\hat S_2}
=\frac{1}{4\pi}\int_0^\infty \kappa\,d\kappa\;\ln\frac{\det\hat S(\kappa)}{\det\hat S_1(\kappa)\det\hat S_2(\kappa)} .$$
(Переход к полярным координатам в полуплоскости $(\xi,k_z)$: $\int d\xi\,dk_z\to\pi\int\kappa d\kappa$.) Деление на $\det\hat S_1\det\hat S_2$ — вычитание собственной энергии каждого крыла: остаётся энергия **взаимодействия** крыльев. Здесь бесконечности сокращаются сами, обрезание $k_m$ не нужно.

### II.3 Сила на каждый элемент
Пусть элемент $j$ поверхности сдвигается на $\delta\mathbf X_j$. Сила на него $\mathbf F_j=-\partial\mathcal E/\partial\mathbf X_j$, и по $\delta\ln\det\hat S=\mathrm{Tr}(\hat S^{-1}\delta\hat S)$
$$\mathbf F_j=-\frac{1}{4\pi}\int_0^\infty\kappa\,d\kappa\;\Big[\mathrm{Tr}\big(\hat S^{-1}\partial_{\mathbf X_j}\hat S\big)-\mathrm{Tr}\big(\hat S_{1,2}^{-1}\partial_{\mathbf X_j}\hat S_{1,2}\big)\Big],\qquad
\partial_{\mathbf r}G_0=-\frac{\kappa K_1(\kappa R)}{2\pi}\frac{\mathbf r-\mathbf r'}{R}.$$
Это то же самое, что интеграл скачка тензора натяжений $[\![\langle T_{ik}\rangle]\!]n_k$ по элементу: вариационная сила и поверхностная сила совпадают (это и проверяется ниже на клине). Сила $\mathbf F_j$ — **полная** сила на элемент со стороны вакуумного поля взаимодействия крыльев, включая нормальную и касательную части и сосредоточенные силы на концах.

### II.4 Дискретизация
Крылья разбиваются на $N$ панелей со сгущением к концам (косинусная сетка); на каждой панели $\sigma=\mathrm{const}$ (коллокация в центрах). Диагональ: $\frac1\pi\int_0^{h/2}K_0(\kappa t)dt$ с аналитическим выделением логарифма. Интеграл по $\kappa$ — квадратура на полуоси с заменой $\kappa=s\,\mathrm{tg}(\pi u/2)$.

### II.5 Что считается результатом
1. Нормальное давление на середину граней — сравнение с Дойчем–Канделасом (конформный скаляр, (5.42); на дирихле‑границе нормальное давление от выбора связи не зависит).
2. Раскладка осевой силы $F_y$ по зонам: внутренние концы у вершины, грани, внешние концы.
3. Полная сила на жёсткий уголок.

**Важно.** $\mathcal E$ зависит только от взаимных расстояний между точками крыльев, поэтому при сдвиге уголка **как целого** $\mathcal E$ не меняется, и $\sum_j\mathbf F_j=0$ тождественно — это следствие трансляционной инвариантности вакуума (теорема Нётер для тела целиком), а не численная находка. Нетривиальный вопрос — **где** приложены силы, уравновешивающие давление на грани, найденное Дойчем–Канделасом и Антипиным.

In [ ]:
EG = 0.5772156649015329
def K0K1(x):
    x = np.asarray(x, float); K0 = np.empty_like(x); K1 = np.empty_like(x)
    s = x <= 2.0; t = x[s]/2; t2 = (x[s]/3.75)**2
    I0 = 1 + t2*(3.5156229 + t2*(3.0899424 + t2*(1.2067492 + t2*(0.2659732 + t2*(0.0360768 + t2*0.0045813)))))
    I1 = x[s]*(0.5 + t2*(0.87890594 + t2*(0.51498869 + t2*(0.15084934 + t2*(0.02658733 + t2*(0.00301532 + t2*0.00032411))))))
    y = t*t
    K0[s] = -np.log(t)*I0 + (-0.57721566 + y*(0.42278420 + y*(0.23069756 + y*(0.03488590 + y*(0.00262698 + y*(0.00010750 + y*0.0000074))))))
    K1[s] = np.log(t)*I1 + (1/x[s])*(1 + y*(0.15443144 + y*(-0.67278579 + y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
    b = ~s; xb = x[b]; z = 2/xb; e = np.exp(-xb)/np.sqrt(xb)
    K0[b] = e*(1.25331414 + z*(-0.07832358 + z*(0.02189568 + z*(-0.01062446 + z*(0.00587872 + z*(-0.00251540 + z*0.00053208))))))
    K1[b] = e*(1.25331414 + z*(0.23498619 + z*(-0.03655620 + z*(0.01504268 + z*(-0.00780353 + z*(0.00325614 + z*(-0.00068245)))))))
    return K0, K1
for xv in (0.01, 0.5, 1.9, 2.1, 5.0, 20.0):
    tt = np.linspace(0, 12, 200001)
    k0i = np.trapezoid(np.exp(-xv*np.cosh(tt)), tt); k1i = np.trapezoid(np.cosh(tt)*np.exp(-xv*np.cosh(tt)), tt)
    a_, b_ = K0K1(np.array([xv]))
    check('K0, K1 (полиномы Абрамовица–Стиган) при x=%.2f' % xv, max(abs(a_[0]/k0i-1), abs(b_[0]/k1i-1)), 2e-7)

In [ ]:
def k0_selfpanel(kap, h):
    xg, wg = np.polynomial.legendre.leggauss(16)
    T = h/2; t = (xg+1)/2*T; w = wg/2*T
    K0, _ = K0K1(kap*t)
    reg = K0 + np.log(kap*t/2)
    return (np.sum(w*reg) - (T*np.log(kap*T/2) - T)) / pi

def panels(P0, P1, N):
    u = (1 - np.cos(np.linspace(0, pi, N+1))) / 2
    P0 = np.asarray(P0, float); P1 = np.asarray(P1, float)
    nodes = P0[None, :] + u[:, None]*(P1 - P0)[None, :]
    return (nodes[1:] + nodes[:-1])/2, np.linalg.norm(nodes[1:] - nodes[:-1], axis=1)

def Smat(mid, h, kap):
    R = np.linalg.norm(mid[:, None, :] - mid[None, :, :], axis=2); np.fill_diagonal(R, 1.0)
    K0, _ = K0K1(kap*R); S = K0/(2*pi)*h[None, :]
    for j in range(len(h)): S[j, j] = k0_selfpanel(kap, h[j])
    return S

def grad_lndet(mid, h, k):
    S = Smat(mid, h, k); Si = np.linalg.inv(S)
    d = mid[:, None, :] - mid[None, :, :]; R = np.linalg.norm(d, axis=2); np.fill_diagonal(R, 1.0)
    _, K1 = K0K1(k*R); np.fill_diagonal(K1, 0.0)
    g = (k*K1/(2*pi*R))[..., None]*d
    G = np.einsum('ji,ijc->jc', Si, g)*h[:, None] + np.einsum('ij,ijc,i->jc', Si, g, h)
    return G, np.linalg.slogdet(S)[1]

def kgrid(n=60, s=1.0):
    u = (np.arange(n) + 0.5)/n
    return s*np.tan(pi/2*u), s*pi/2/np.cos(pi/2*u)**2/n

def casimir(W1, W2, nk=60, ks=3.0):
    (m1, h1), (m2, h2) = W1, W2
    mid = np.vstack([m1, m2]); h = np.concatenate([h1, h2]); N1 = len(h1)
    kk, ww = kgrid(nk, ks); F = np.zeros((len(h), 2)); E = 0.0
    for k, w in zip(kk, ww):
        G, L = grad_lndet(mid, h, k); G1, L1 = grad_lndet(m1, h1, k); G2, L2 = grad_lndet(m2, h2, k)
        G[:N1] -= G1; G[N1:] -= G2
        F += -(1/(4*pi))*w*k*G
        E += w*k*(L - L1 - L2)
    return E/(4*pi), F, N1

## Часть III. Численное решение

### III.1 Проверка метода на двух параллельных пластинах
Полосы ширины $W$ на расстоянии $a=1$. Для бесконечных пластин (дирихле‑скаляр в 3D): $\mathcal E/W=-\pi^2/1440$, давление $\pi^2/480$. Для конечной $W$ поправка от краёв $\propto 1/W$.

Допуск 3 %: это погрешность дискретизации (см. сходимость по $N$ в III.4 — грани меняются на ~2 % при $N=200\to280$) плюс неполная экстраполяция краевых поправок полосы. Для качественных выводов этого достаточно; уточнение — увеличением $N$.

In [ ]:
res = []
for Wd, N in [(10, 120), (20, 200)]:
    W1 = panels([-Wd/2, 0], [Wd/2, 0], N); W2 = panels([-Wd/2, 1.0], [Wd/2, 1.0], N)
    E, F, N1 = casimir(W1, W2, nk=40, ks=0.5)
    res.append((Wd, E/Wd, F[:N1, 1].sum()/Wd))
    print('W=%2d   E/W = %.6f  (беск.: %.6f)    давление = %.6f  (беск.: %.6f)    F1+F2 = %.1e'
          % (Wd, E/Wd, -pi**2/1440, F[:N1, 1].sum()/Wd, pi**2/480, F[:N1, 1].sum() + F[N1:, 1].sum()))
# экстраполяция по 1/W
(W1_, e1, p1), (W2_, e2, p2) = res
e_inf = (W2_*e2 - W1_*e1)/(W2_ - W1_); p_inf = (W2_*p2 - W1_*p1)/(W2_ - W1_)
check('пластины: E/W -> -pi^2/1440 (экстраполяция 1/W)', e_inf/(-pi**2/1440) - 1, 3e-2)
check('пластины: давление -> pi^2/480 (экстраполяция 1/W)', p_inf/(pi**2/480) - 1, 3e-2)

### III.2 Давление на середину граней против Дойча–Канделаса
Конформный скаляр в клине с углом $\theta$ (Дойч–Канделас (5.42)): $f_s(\theta)=\dfrac{\pi^2}{2\theta^2}\Big(\dfrac{\pi^2}{\theta^2}-\dfrac{\theta^2}{\pi^2}\Big)$, давление на грань $\dfrac{f_s(\theta)-f_s(2\pi-\theta)}{240\pi^2r^4}$ (изнутри минус снаружи; нормировка — на скаляр в этих единицах).

In [ ]:
fS = lambda th: (pi**2/(2*th**2))*(pi**2/th**2 - th**2/pi**2)
def vangle(alpha, r0, r1, N, nk=60, ks=3.0):
    u1 = np.array([np.sin(alpha), np.cos(alpha)]); u2 = np.array([-np.sin(alpha), np.cos(alpha)])
    W1 = panels(r0*u1, r1*u1, N); W2 = panels(r0*u2, r1*u2, N)
    E, F, N1 = casimir(W1, W2, nk, ks)
    return W1, W2, E, F, N1
alpha = np.radians(15)
t0 = time.time(); W1, W2, E, F, N1 = vangle(alpha, 0.1, 4.0, 200); print('время %.1f с' % (time.time() - t0))
r = np.linalg.norm(W1[0], axis=1); h1 = W1[1]
nin = np.array([-np.cos(alpha), np.sin(alpha)])          # внутренняя нормаль крыла 1
dens = (F[:N1] @ nin)/h1
pred = (fS(2*alpha) - fS(2*pi - 2*alpha))/(240*pi**2*r**4)
worst = 0
for rr in (0.5, 0.8, 1.0, 1.5, 2.0, 3.0):
    i = np.argmin(abs(r - rr)); worst = max(worst, abs(dens[i]/pred[i] - 1))
    print('r = %.2f   численно %.4e   Дойч–Канделас %.4e   отношение %.4f' % (r[i], dens[i], pred[i], dens[i]/pred[i]))
check('давление на грань = Дойч–Канделас (середина граней, r=0.5..3)', worst, 3e-2)

### III.3 Раскладка осевой силы по зонам
$F_y>0$ — к раствору уголка («тяга Антипина»). Зоны по расстоянию от вершины: **внутренние концы** $r<3r_0$, **грани** $3r_0<r<r_1-0.3$, **внешние концы** $r>r_1-0.3$. Рядом — прогноз для граней по Дойчу–Канделасу: $F_y^{\rm DC}=\dfrac{2\sin\alpha\,[f_s(2\alpha)-f_s(2\pi-2\alpha)]}{720\pi^2}\Big(\dfrac1{(3r_0)^3}-\dfrac1{(r_1-0.3)^3}\Big)$.

In [ ]:
r0, r1 = 0.1, 4.0
print(' alpha   F_y полн.     внутр. концы    грани        [ДК для граней]   внешн. концы')
for ad in (10, 30, 45):
    a = np.radians(ad); W1, W2, E, F, N1 = vangle(a, r0, r1, 200)
    r = np.linalg.norm(W1[0], axis=1); Fy = F[:N1, 1] + F[N1:, 1]
    vi = r < 3*r0; to = r > r1 - 0.3; fa = ~(vi | to)
    dc = 2*np.sin(a)*(fS(2*a) - fS(2*pi - 2*a))/(720*pi**2)*(1/(3*r0)**3 - 1/(r1 - 0.3)**3)
    print('%4d°   %+.1e    %+.5e    %+.5e   [%+.5e]     %+.2e' % (ad, Fy.sum(), Fy[vi].sum(), Fy[fa].sum(), dc, Fy[to].sum()))
    check('alpha=%d°: полная сила на жёсткий уголок = 0 (относительно граней)' % ad, Fy.sum()/Fy[fa].sum(), 1e-6)
    check('alpha=%d°: грани ~ Дойч–Канделас' % ad, Fy[fa].sum()/dc - 1, 0.12)

### III.4 Сходимость и масштабирование с зазором у вершины
Если компенсация граней — свойство вершины, то сила на внутренние концы должна расти так же, как сила на грани, $\propto r_0^{-3}$, и их отношение стремиться к $-1$.

In [ ]:
alpha = np.radians(15)
for r0, N in [(0.2, 140), (0.1, 200), (0.05, 260)]:
    W1, W2, E, F, N1 = vangle(alpha, r0, 4.0, N)
    r = np.linalg.norm(W1[0], axis=1); Fy = F[:N1, 1] + F[N1:, 1]
    face = Fy[(r > 3*r0) & (r < 3.7)].sum(); vert = Fy[r <= 3*r0].sum()
    print('r0 = %.2f  N = %3d   грани %+.4e   внутр. концы %+.4e   -концы/грани = %.4f   r0^3*грани = %.4f' % (r0, N, face, vert, -vert/face, r0**3*face))
for N in (120, 200, 280):
    W1, W2, E, F, N1 = vangle(alpha, 0.1, 4.0, N)
    r = np.linalg.norm(W1[0], axis=1); Fy = F[:N1, 1] + F[N1:, 1]
    print('N = %3d   грани %+.5f   E = %.6f' % (N, Fy[(r > 0.3) & (r < 3.7)].sum(), E))

## Часть IV. Контур 1: поток тензора натяжений вокруг всего уголка (без вариации энергии)

Здесь сила **не** вычисляется как $-\partial\mathcal E/\partial X$. Берётся только поле в вакууме и тензор:
$$\vec F=\oint_{S_R}\langle T_{ik}\rangle\,n_k\,dl ,$$
где $S_R$ — окружность радиуса $R$, охватывающая уголок целиком (рис. `Antipin_angle_tensor_contours.png`).

**Тензор из функции Грина.** Для скаляра Дирихле на мнимой частоте отражённая часть функции Грина (поле от индуцированного простого слоя)
$$G_s(\mathbf r,\mathbf r';\kappa)=-\sum_{b,b'}G_0(\mathbf r,\mathbf s_b)h_b\,[S^{-1}]_{bb'}\,G_0(\mathbf s_{b'},\mathbf r') .$$
Перенормированный тензор (вычтен вакуум без тела) в точке $\mathbf r$:
$$\langle T_{ij}\rangle=-\frac{1}{4\pi^2}\int_0^\infty\kappa\,d\kappa\,\Big[\partial_i\partial'_j G_s-\tfrac12\delta_{ij}\big(\nabla\!\cdot\!\nabla' G_s+\kappa^2G_s\big)\Big]_{\mathbf r'=\mathbf r},$$
(множитель $\int d\xi\,dk_z\to\pi\int\kappa\,d\kappa$, как в части II; знак — сила на тела внутри контура).

**Проверки.**
1. *Метод контура верен:* два разнесённых тела A и B; поток тензора по окружности вокруг одного A должен совпасть с силой на A, найденной в части III. Если совпадает — формула тензора и нормировка правильны.
2. *Сохранение в вакууме:* поток вокруг всего уголка не должен зависеть от $R$.
3. *Ответ:* $F_x,F_y$ по контуру вокруг всего уголка.

In [ ]:
def G0grad(r, s, k):
    d = r[:, None, :] - s[None, :, :]; R = np.linalg.norm(d, axis=2)
    K0, K1 = K0K1(k*R)
    return K0/(2*pi), (-k*K1/(2*pi*R))[..., None]*d

def contour_force(bodies, center, Rc, M=300, nk=60):
    mid = np.vstack([b[0] for b in bodies]); h = np.concatenate([b[1] for b in bodies])
    th = 2*pi*np.arange(M)/M; n = np.stack([np.cos(th), np.sin(th)], 1)
    pts = center + Rc*n; dl = 2*pi*Rc/M
    dmin = np.min(np.linalg.norm(pts[:, None, :] - mid[None, :, :], axis=2))
    kk, ww = kgrid(nk, 1.0/dmin); F = np.zeros(2)
    for k, w in zip(kk, ww):
        S = Smat(mid, h, k); G, gr = G0grad(pts, mid, k)
        X0 = np.linalg.solve(S, G.T)
        Xg = np.stack([np.linalg.solve(S, gr[:, :, j].T) for j in range(2)], -1)
        Gs = -np.einsum('mb,bm->m', G*h[None, :], X0)
        Dij = -np.einsum('mbi,bmj->mij', gr*h[None, :, None], Xg)
        Theta = Dij - 0.5*np.eye(2)[None]*(k*k*Gs + Dij[:, 0, 0] + Dij[:, 1, 1])[:, None, None]
        F += w*k/(2*pi)*np.einsum('mij,mj->i', Theta, n)*dl
    return -F

# 1) проверка метода на двух разнесённых телах
A = panels([-0.5, 0], [0.5, 0], 60); B = panels([1.5, -0.3], [1.9, 0.6], 40)
E, Fv, NA = casimir(A, B, nk=50)
FA = Fv[:NA].sum(0)
print('сила на A (часть III):        Fx = %+.6e  Fy = %+.6e' % tuple(FA))
for Rc in (0.8, 1.0):
    Ft = contour_force([A, B], np.array([0.0, 0.0]), Rc, M=300, nk=50)
    print('поток тензора вокруг A, R=%.1f: Fx = %+.6e  Fy = %+.6e' % ((Rc,) + tuple(Ft)))
    check('поток тензора вокруг A = сила на A (R=%.1f)' % Rc, np.linalg.norm(Ft - FA)/np.linalg.norm(FA), 2e-3)

In [ ]:
# 2–3) контур вокруг всего уголка
alpha = np.radians(15); r0, r1 = 0.3, 3.0
u1 = np.array([np.sin(alpha), np.cos(alpha)]); u2 = np.array([-np.sin(alpha), np.cos(alpha)])
W1 = panels(r0*u1, r1*u1, 100); W2 = panels(r0*u2, r1*u2, 100)
E, Fv, N1 = casimir(W1, W2, nk=50)
scale = np.abs(Fv[:N1, 0].sum())      # характерная сила: сжатие крыльев друг к другу
print('масштаб: сила сжатия на одно крыло |Fx| = %.4f' % scale)
cen = np.array([0, 0.5*(r0 + r1)*np.cos(alpha)])
for Rc in (2.2, 3.0, 4.5, 7.0):
    Ft = contour_force([W1, W2], cen, Rc, M=480, nk=60)
    print('R = %.1f   Fx = %+.2e   Fy = %+.2e   |F|/масштаб = %.1e' % (Rc, Ft[0], Ft[1], np.linalg.norm(Ft)/scale))
    check('поток тензора вокруг всего уголка = 0 (R=%.1f)' % Rc, np.linalg.norm(Ft)/scale, 1e-6)

**Итог части IV.**
* Метод контура проверен: поток тензора вокруг одного тела совпадает с силой на него из части III с точностью $\sim3\cdot10^{-4}$.
* Поток тензора Максвелла по окружности, охватывающей весь уголок, равен нулю при всех $R$ (относительно силы сжатия крыльев — $10^{-11}\ldots10^{-14}$, на уровне машинной точности).
* Это независимо от принципа виртуальной работы: используются только поля в вакууме и $\langle T_{ik}\rangle$. Результирующей силы на уголок в этой модели (2D, скаляр Дирихле, $T=0$, равновесие) нет.

**Что этим не охвачено:** ЭМ поле (TE‑поляризация), реальный металл $\varepsilon(\omega)$, конечная температура и неравновесные состояния, движение уголка. Ответ относится к статическому вакуумному состоянию; вопрос о динамике (виманик, MenDrive) им не закрывается.

## Итог

1. **Давление на грани уголка реально** и совпадает с точным решением Дойча–Канделаса с точностью ~1 % на середине граней (интегрально по граням — в пределах 0,1–8 % в зависимости от угла и дискретизации): сила действует по нормали к граням и внутрь уголка, её осевая часть направлена к раствору — это то, что утверждают Антипин и приложение D (с поправкой коэффициента, часть I).
2. **Эта осевая сила полностью уравновешена силами на внутренних концах крыльев у вершины.** Отношение «концы / грани» стремится к $-1$ при уменьшении зазора $r_0$ (1,0046 → 1,0005 → 1,0001), и обе силы растут как $r_0^{-3}$. Внешние концы крыльев дают пренебрежимо малый вклад.
3. **Полная сила на жёсткий уголок равна нулю.** В рамках метода это тождество (энергия зависит только от взаимных расстояний); физическое содержание расчёта — в п. 2: показано, *где именно* компенсируется давление на грани.

**Почему PFA этого не видит.** PFA строит давление из модели «плоскопараллельных пластин» в каждой точке грани и не содержит сосредоточенных сил на концах крыльев, где модель пластин заведомо неверна. Именно у вершины, где у Антипина интеграл расходится как $L_{\min}^{-3}$, точный расчёт даёт равную и противоположную силу.

**Ограничения.** 2D (уголок бесконечен вдоль оси), одна поляризация (TM = Дирихле; TE = Нейман следует добавить для полного ЭМ ответа, но аргумент п. 3 от поляризации не зависит), идеальный проводник, нулевая толщина крыльев, крылья у вершины разомкнуты ($r_0>0$). Для сомкнутой вершины ($r_0\to0$) тело одно; по п. 3 полная сила на него также нулевая.